In [ ]:
import os
import cv2
import numpy as np
import shutil
from tqdm import tqdm
from sklearn.cluster import KMeans
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array

from google.colab import drive
drive.mount('/content/drive')

IMAGE_PATH = '/content/drive/MyDrive/uvillas_hojas_enfermas'
OUTPUT_PATH = '/content/drive/MyDrive/Uvillas_hojas_enfermas_clusterizadas'

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load AI model (feature extractor)
print(" Loading MobileNetV2...")
model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')

# Read files
files = [f for f in os.listdir(IMAGE_PATH)
         if f.lower().endswith(('.jpg','.jpeg','.png'))]
features = []
valid_files = []
print("Analyzing images with AI...")
for file in tqdm(files):
    path = os.path.join(IMAGE_PATH, file)
    img = cv2.imread(path)
    if img is None:
        continue
    img = cv2.resize(img, (224, 224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    feat = model.predict(img, verbose=0)[0]
    features.append(feat)
    valid_files.append(file)
X = np.array(features)

print("Clustering into 5 classes with KMeans...")
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
# Create folders
for i in range(5):
    os.makedirs(os.path.join(OUTPUT_PATH, f"class_{i+1}"), exist_ok=True)

print("Moving images...")
for file, label in zip(valid_files, labels):
    source = os.path.join(IMAGE_PATH, file)
    destination = os.path.join(OUTPUT_PATH, f"class_{label+1}", file)
    shutil.copy(source, destination)

print("✅ Classification completed")
print("Check the folder:", OUTPUT_PATH)
